# Estimating Park-and-Ride Lot Choice

This notebook estimates the single-segment park-and-ride lot choice model with Larch. The model is an interaction model, so it uses chooser-alternative expression values from the ActivitySim estimation data bundle.

Run ActivitySim in estimation mode first. The estimation run must include the park-and-ride fixes that preserve `pnr_zone_id` and configure it as the alternative ID.

## Load libraries

In [17]:
from pathlib import Path
from types import SimpleNamespace

import larch as lx
import pandas as pd

from activitysim.estimation.larch.general import (
    apply_coefficients,
    linear_utility_from_spec,
    update_coefficients,
)

lx.versions()

{'larch': '6.0.46',
 'sharrow': '2.16.2',
 'numpy': '2.2.6',
 'pandas': '2.3.2',
 'xarray': '2025.6.1',
 'numba': '0.62.0',
 'jax': 'not installed'}

## Locate the estimation data bundle

Change `EDB_DIRECTORY` if your ActivitySim output directory is elsewhere.

In [18]:
modelname = "park_and_ride_lot_choice"
#EDB_DIRECTORY = Path("output-est-mode/estimation_data_bundle/{modelname}/")
EDB_DIRECTORY = Path(r"D:\stefan\asim_estimation_pnr\outputs_current\estimation_data_bundle\park_and_ride_lot_choice")


In [19]:
def read_edb_file(filename, **kwargs):
    path = EDB_DIRECTORY / filename
    parquet_path = path.with_suffix(".parquet")
    if parquet_path.exists():
        return pd.read_parquet(parquet_path, **kwargs)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path.name}. Re-run ActivitySim estimation after setting "
            f"the park-and-ride chooser ID, then coalesce the interaction EDB."
        )
    return pd.read_csv(path, **kwargs)

coefficient_candidates = [
    f"{modelname}_coefficients.csv",
    f"{modelname}_{modelname}_coeffs.csv",
    f"{modelname}_coeffs.csv",
]
coefficient_file = next(
    (filename for filename in coefficient_candidates if (EDB_DIRECTORY / filename).exists()),
    None,
)
if coefficient_file is None:
    raise FileNotFoundError(
        f"Could not find PNR coefficients in {EDB_DIRECTORY}. "
        f"Expected one of: {coefficient_candidates}"
    )

coefficients = read_edb_file(coefficient_file, index_col="coefficient_name")
spec = read_edb_file(f"{modelname}_SPEC.csv", comment="#")
alternatives = read_edb_file(f"{modelname}_alternatives_combined.csv")
choosers = read_edb_file(f"{modelname}_choosers_combined.csv")

if "pnr_zone_id" not in alternatives.columns:
    raise KeyError(f"Expected 'pnr_zone_id' in alternatives: {alternatives.columns.tolist()}")
chooser_id_column = choosers.columns[0]
if chooser_id_column not in alternatives.columns:
    raise ValueError(
        f"The alternatives EDB is missing chooser ID {chooser_id_column!r}. "
        "Re-run estimation after setting estimator.set_chooser_id(...) in the "
        "park-and-ride model."
    )

spec = spec.loc[spec.Expression.notna()].copy()
spec = spec.loc[~spec.Expression.str.startswith("_")].copy()
print(f"Using coefficient file: {coefficient_file}")
coefficients.head(), spec.head(), alternatives.head(), choosers.head()

Using coefficient file: park_and_ride_lot_choice_coefficients.csv


(                         value constrain
 coefficient_name                        
 coef_zero             0.000000         T
 coef_unavailable   -999.000000         F
 coef_ivt_auto        -0.077806         F
 coef_ivt_transit     -0.048312         F
 coef_auto_op_costs    0.000000         F,
               Label                               Description  \
 4   util_drive_time  round trip drive time from origin to lot   
 5  util_drive_tolls                                toll costs   
 6   util_first_wait                           first wait time   
 7    util_xfer_wait                        transfer wait time   
 8   util_total_wait                           total wait time   
 
                                           Expression         coefficient  
 4     @_d_pnr_time_skims_out + _d_pnr_time_skims_inb       coef_ivt_auto  
 5     @_d_pnr_cost_skims_out + _d_pnr_cost_skims_inb      coef_toll_auto  
 6     @(ldt_skims['iwtwpnr'] + dlt_skims['iwtwpnr'])   coef_wait_transit  
 7 

In [20]:
# Optional: exclude survey tours with a ~$17.90 toll on the home-to-lot leg (likely ferry); the toll is not modeled
# EXCLUDED_TOUR_IDS = [10431751, 9927843]
# alternatives = alternatives.loc[~alternatives[chooser_id_column].isin(EXCLUDED_TOUR_IDS)].copy()
# choosers = choosers.loc[~choosers[chooser_id_column].isin(EXCLUDED_TOUR_IDS)].copy()

## Build the Larch interaction model

The first two columns of `alternatives_combined` identify the chooser and alternative. The chooser table contains the observed choice written by ActivitySim. PNR uses `pnr_zone_id` as the alternative ID.

In [21]:
########Build the interaction model
alternative_id_column = "pnr_zone_id"

if alternative_id_column not in alternatives.columns:
    raise KeyError(f"Expected {alternative_id_column!r} in alternatives: {alternatives.columns.tolist()}")

key_columns = [chooser_id_column, alternative_id_column]
duplicate_count = int(alternatives.duplicated(key_columns).sum())
if duplicate_count:
    print(f"Removing {duplicate_count} duplicate chooser-alternative rows from the EDB")
    alternatives = alternatives.drop_duplicates(key_columns, keep="last").copy()

x_ca = alternatives.set_index(key_columns)
choosers = choosers.drop_duplicates(chooser_id_column, keep="last").copy()
x_co = choosers.loc[choosers[chooser_id_column].isin(alternatives[chooser_id_column])].set_index(chooser_id_column)

print(f"CA index: {x_ca.index.names}, unique={x_ca.index.is_unique}, rows={len(x_ca)}")
print(f"CO index: {x_co.index.name}, unique={x_co.index.is_unique}, rows={len(x_co)}")

choice_candidates = ["override_choice", "model_choice", "choice", "pnr_zone_id"]
choice_column = next((c for c in choice_candidates if c in x_co.columns), None)
if choice_column is None:
    raise KeyError(f"Could not find an observed choice column in {x_co.columns.tolist()}")

# Keep constant chooser-alternative variables two-dimensional for Larch.
d_ca = lx.Dataset.construct.from_idca(x_ca, crack=False)
d_co = lx.Dataset.construct.from_idco(x_co)
datatree = d_ca.merge(d_co)

model = lx.Model(datatree=datatree, compute_engine="numba")
alt_codes = datatree.dc.altids()
root_id = -1 if 0 in alt_codes else 0
model.initialize_graph(alternative_codes=alt_codes, root_id=root_id)

model.utility_ca = linear_utility_from_spec(
    spec,
    x_col="Label",
    p_col="coefficient",
    ignore_x=("#",),
    x_validator=set(x_ca.columns),
    expr_col="Expression",
)
apply_coefficients(coefficients, model)
model.choice_co_code = choice_column
model.availability_any = True
model.set_cap(50)

model.required_data()

CA index: ['tour_id', 'pnr_zone_id'], unique=True, rows=5733
CO index: tour_id, unique=True, rows=39


key,value
ca,"['util_drive_time', 'util_drive_tolls', 'util_first_wait', 'util_ivtt', 'util_log_lot_capacity', 'util_percent_drive', 'util_total_wait', 'util_transfer_penalty', 'util_walk_time', 'util_xfer_wait']"
choice_co_code,'override_choice'
avail_any,True


## Review and diagnose the model

In [22]:
model, problems = model.doctor()
problems

<empty>


## Estimate

In [23]:
model.estimate(method="BHHH", options={"maxiter": 1000})

,value,best,initvalue,minimum,maximum,nullvalue,holdfast
param_name,,,,,,,
coef_ivt_auto,-0.064196,-0.064196,-0.077806,-50.0,50.0,0.0,0
coef_ivt_transit,-0.069902,-0.069902,-0.048312,-50.0,50.0,0.0,0
coef_lot_capacity,0.473761,0.473761,0.662689,-50.0,50.0,0.0,0
coef_percent_drive,-5.823242,-5.823242,0.000000,-50.0,50.0,0.0,0
coef_toll_auto,0.221047,0.221047,0.000000,-50.0,50.0,0.0,0
coef_wait_transit,-0.017792,-0.017792,-0.044691,-50.0,50.0,0.0,0
coef_walk_transit,-0.043890,-0.043890,-0.112051,-50.0,50.0,0.0,0
coef_xfer_transit,-0.205820,-0.205820,0.000000,-50.0,50.0,0.0,0
coef_xwait_transit,-0.005215,-0.005215,0.000000,-50.0,50.0,0.0,0


┣          loglike: np.float64(-85.95625856645239)
┣                x: coef_ivt_auto        -0.064196
┃                   coef_ivt_transit     -0.069902
┃                   coef_lot_capacity     0.473761
┃                   coef_percent_drive   -5.823242
┃                   coef_toll_auto        0.221047
┃                   coef_wait_transit    -0.017792
┃                   coef_walk_transit    -0.043890
┃                   coef_xfer_transit    -0.205820
┃                   coef_xwait_transit   -0.005215
┃                   dtype: float64
┣        tolerance: np.float64(7.46270403264644e-06)
┣            steps: array([1.        , 1.        , 0.5       , 0.5       , 0.5       ,
┃                          0.5       , 0.25      , 0.1125    , 0.3725    , 0.1735    ,
┃                          0.14085   , 0.41974   , 0.243818  , 0.1350408 , 0.44517952,
┃                          0.13846283, 0.27644823, 0.12389494, 0.44761053, 0.1431596 ,
┃                          0.22591523, 0.24340571, 0.2367972 , 0.12968883, 0.39158663,
┃                          0.24547872, 0.24939142, 0.12529428, 0.45657595, 0.1468327 ,
┃                          0.24471461, 0.24456179, 0.24359586, 0.13362809, 0.40533322,
┃                          0.25436672, 0.25629714, 0.1293221 , 0.47157891, 0.15168981,
┃                          0.25265093, 0.25230778, 0.25150991, 0.13797373, 0.41845286,
┃                          0.26257904, 0.13228233, 0.48111915, 0.14324071, 0.57506964,
┃                          0.15942909, 0.29822819, 0.16570868, 0.26833526, 0.29335417,
┃                          0.23701108, 0.12626374, 0.43626917, 0.27224668, 0.13651448,
┃                          0.48332206, 0.14546161, 0.5895256 , 0.16270704, 0.30350616,
┃                          0.16845225, 0.27393053, 0.14981216, 0.42336326, 0.13190644])
┣          message: 'Optimization terminated successfully'
┣     elapsed_time: datetime.timedelta(seconds=1, microseconds=152258)
┣           method: 'BHHH'
┣          n_cases: 39
┣ iteration_number: 70

In [24]:
model.parameter_summary()
model.estimation_statistics()

<xmle.Elem 'div' with 1 children>

## Write estimation results

In [25]:
result_dir = EDB_DIRECTORY / "estimated"
data = SimpleNamespace(edb_directory=EDB_DIRECTORY, coefficients=coefficients)

update_coefficients(
    model,
    data,
    result_dir,
    output_file=f"{modelname}_coefficients.csv",
)

model.to_xlsx(
    result_dir / f"{modelname}_model_estimation.xlsx",
    data_statistics=False,
)

In [26]:
model.parameter_summary()

,Value,Std Err,t Stat,Signif,Null Value
Parameter,,,,,
coef_ivt_auto,-0.0642,0.0157,-4.10,***,0.00
coef_ivt_transit,-0.0699,0.0167,-4.17,***,0.00
coef_lot_capacity,0.474,0.182,2.60,**,0.00
coef_percent_drive,-5.82,2.37,-2.46,*,0.00
coef_toll_auto,0.221,0.0839,2.63,**,0.00
coef_wait_transit,-0.0178,0.0125,-1.42,,0.00
coef_walk_transit,-0.0439,0.0202,-2.17,*,0.00
coef_xfer_transit,-0.206,0.355,-0.58,,0.00
coef_xwait_transit,-0.00522,0.0291,-0.18,,0.00


## Next steps

Review the revised coefficient file, then copy `park_and_ride_lot_choice_coefficients.csv` into the active configuration directory before running ActivitySim in simulation mode.

In [27]:
pd.read_csv(result_dir / f"{modelname}_coefficients.csv")

,coefficient_name,value,constrain
0,coef_zero,0.000000,T
1,coef_unavailable,-999.000000,F
2,coef_ivt_auto,-0.064196,F
3,coef_ivt_transit,-0.069902,F
4,coef_auto_op_costs,0.000000,F
5,coef_walk_transit,-0.043890,F
6,coef_wait_transit,-0.017792,F
7,coef_xfer_transit,-0.205820,F
8,coef_xwait_transit,-0.005215,F
9,coef_small_lot,0.000000,F


In [28]:
# Compare mean predictor values for the chosen alternative vs. all alternatives
predictor_cols = [col for col in dict.fromkeys(spec["Label"]) if col in x_ca.columns]

chosen_by_chooser = x_co[choice_column].to_dict()
alt_ids = x_ca.index.get_level_values(alternative_id_column)
chooser_ids = x_ca.index.get_level_values(chooser_id_column)
is_chosen = alt_ids == chooser_ids.map(chosen_by_chooser)

comparison = pd.DataFrame(
    {
        "Chosen Alt Mean": x_ca.loc[is_chosen, predictor_cols].mean(),
        "All Alts Mean": x_ca[predictor_cols].mean(),
    }
)
comparison["Difference"] = comparison["Chosen Alt Mean"] - comparison["All Alts Mean"]
comparison.index.name = "Variable"
comparison

,Chosen Alt Mean,All Alts Mean,Difference
Variable,,,
util_drive_time,35.137123,112.517128,-77.380005
util_drive_tolls,0.457632,1.963738,-1.506105
util_first_wait,21.112215,25.060415,-3.948200
util_xfer_wait,11.286400,34.743252,-23.456852
util_total_wait,32.398613,59.803665,-27.405052
util_walk_time,25.221519,41.669800,-16.448280
util_ivtt,56.971474,120.209511,-63.238037
util_transfer_penalty,1.025727,2.760425,-1.734697
util_log_lot_capacity,6.196738,5.187129,1.009609


In [29]:
x_ca.loc[is_chosen, predictor_cols]

,,util_drive_time,util_drive_tolls,util_first_wait,util_xfer_wait,util_total_wait,util_walk_time,util_ivtt,util_transfer_penalty,util_log_lot_capacity,util_percent_drive
tour_id,pnr_zone_id,,,,,,,,,,
9250500,66236,18.459089,0.000000,31.008402,19.622068,50.630470,32.576805,111.647537,1.714286,7.733684,0.141877
4183761,66236,16.661467,0.000000,12.919355,0.000000,12.919355,13.617260,116.280869,0.000000,7.733684,0.125329
4872110,66241,65.951553,0.000000,11.834758,0.000000,11.834758,4.989394,4.994812,0.000000,6.904751,0.929597
3838500,66242,62.115025,0.000000,26.250000,18.460346,44.710346,19.764475,91.464699,1.333333,6.456770,0.404448
11251300,66248,33.673740,0.000000,26.785713,0.000000,26.785713,11.519560,39.772682,0.000000,5.780744,0.458480
1899200,66260,22.325586,0.000000,22.000000,0.000000,22.000000,19.931280,39.917713,0.000000,5.278115,0.358683
6125890,66268,13.676851,0.000000,14.867724,3.956044,18.823769,22.274820,85.118820,0.857143,6.003887,0.138436
14612890,66276,29.429062,0.000000,14.970414,0.000000,14.970414,79.749962,9.989868,0.000000,6.104793,0.746572
3153800,66276,29.020229,0.000000,18.600618,30.661734,49.262352,15.980379,75.591988,3.455109,6.104793,0.277408
